In [104]:
!pip install neo4j

In [105]:
import os

import tiktoken
from neo4j import GraphDatabase
from openai import OpenAI

neo4j_driver = GraphDatabase.driver("neo4j+s://2e998866.databases.neo4j.io", auth=("2e998866", "XX"))

open_ai_client = OpenAI(
    api_key="XX"
)


def chunk_text(text, chunk_size, overlap, split_on_whitespace_only=True):
    chunks = []
    index = 0

    while index < len(text):
        if split_on_whitespace_only:
            prev_whitespace = 0
            left_index = index - overlap
            while left_index >= 0:
                if text[left_index] == " ":
                    prev_whitespace = left_index
                    break
                left_index -= 1
            next_whitespace = text.find(" ", index + chunk_size)
            if next_whitespace == -1:
                next_whitespace = len(text)
            chunk = text[prev_whitespace:next_whitespace].strip()
            chunks.append(chunk)
            index = next_whitespace + 1
        else:
            start = max(0, index - overlap + 1)
            end = min(index + chunk_size + overlap, len(text))
            chunk = text[start:end].strip()
            chunks.append(chunk)
            index += chunk_size

    return chunks


def num_tokens_from_string(string: str, model: str = "gpt-4") -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.encoding_for_model(model)
    num_tokens = len(encoding.encode(string))
    return num_tokens


def embed(texts, model="text-embedding-3-small"):
    response = open_ai_client.embeddings.create(
        input=texts,
        model=model,
    )
    return list(map(lambda n: n.embedding, response.data))


def chat(messages, model="gpt-4o", temperature=0, config={}):
    response = open_ai_client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=messages,
        **config,
    )
    return response.choices[0].message.content


def tool_choice(messages, model="gpt-4o", temperature=0, tools=[], config={}):
    response = open_ai_client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=messages,
        tools=tools or None,
        **config,
    )
    return response.choices[0].message.tool_calls

In [106]:
import neo4j
from neo4j.exceptions import ClientError
from typing import Any, Optional

NODE_PROPERTIES_QUERY = """
CALL apoc.meta.data()
YIELD label, other, elementType, type, property
WHERE NOT type = "RELATIONSHIP" AND elementType = "node"
WITH label AS nodeLabels, collect({property:property, type:type}) AS properties
RETURN {labels: nodeLabels, properties: properties} AS output
"""

REL_PROPERTIES_QUERY = """
CALL apoc.meta.data()
YIELD label, other, elementType, type, property
WHERE NOT type = "RELATIONSHIP" AND elementType = "relationship"
WITH label AS relType, collect({property:property, type:type}) AS properties
RETURN {type: relType, properties: properties} AS output
"""

REL_QUERY = """
CALL apoc.meta.data()
YIELD label, other, elementType, type, property
WHERE type = "RELATIONSHIP" AND elementType = "node"
UNWIND other AS other_node
RETURN {start: label, type: property, end: toString(other_node)} AS output
"""


def query_database(
    driver: neo4j.Driver, query: str, params: dict[str, Any] = None
) -> list[dict[str, Any]]:
    if params is None:
        params = {}
    data = driver.execute_query(query, params)
    return [r.data() for r in data.records]


def get_schema(
    driver: neo4j.Driver,
) -> str:
    structured_schema = get_structured_schema(driver)

    def _format_props(props: list[dict[str, Any]]) -> str:
        return ", ".join([f"{prop['property']}: {prop['type']}" for prop in props])

    formatted_node_props = [
        f"{label} {{{_format_props(props)}}}"
        for label, props in structured_schema["node_props"].items()
    ]

    formatted_rel_props = [
        f"{rel_type} {{{_format_props(props)}}}"
        for rel_type, props in structured_schema["rel_props"].items()
    ]

    formatted_rels = [
        f"(:{element['start']})-[:{element['type']}]->(:{element['end']})"
        for element in structured_schema["relationships"]
    ]

    return "\n".join(
        [
            "Node properties:",
            "\n".join(formatted_node_props),
            "Relationship properties:",
            "\n".join(formatted_rel_props),
            "The relationships:",
            "\n".join(formatted_rels),
        ]
    )


def get_structured_schema(driver: neo4j.Driver) -> dict[str, Any]:
    node_labels_response = driver.execute_query(NODE_PROPERTIES_QUERY)
    node_properties = [
        data["output"] for data in [r.data() for r in node_labels_response.records]
    ]

    rel_properties_query_response = driver.execute_query(REL_PROPERTIES_QUERY)
    rel_properties = [
        data["output"]
        for data in [r.data() for r in rel_properties_query_response.records]
    ]

    rel_query_response = driver.execute_query(REL_QUERY)
    relationships = [
        data["output"] for data in [r.data() for r in rel_query_response.records]
    ]

    return {
        "node_props": {el["labels"]: el["properties"] for el in node_properties},
        "rel_props": {el["type"]: el["properties"] for el in rel_properties},
        "relationships": relationships,
    }

In [107]:
from importlib import reload
import json

In [108]:
query_update_prompt = """
    Eres un experto en actualizar preguntas para lograr que soliciten una sola cosa, siendo más atómicas, específicas y fáciles de responder.
    Logras esto completando la información faltante en la pregunta con los datos adicionales que te fueron proporcionados en las respuestas anteriores.

    Respondes con la pregunta actualizada que contenga toda la información en ella.
    Solo edita la pregunta si es necesario. Si la pregunta original ya es atómica, específica y fácil de responder, mantén la original.
    No solicites más información de la que pide la pregunta original. Solo reformula la pregunta para hacerla más completa.

    JSON:
    {
        "question": "pregunta1"
    }
"""

def query_update(input: str, answers: list[any]) -> str:
    messages = [
        {"role": "system", "content": query_update_prompt},
        *answers,
        {"role": "user", "content": f"The user question to rewrite: '{input}'"},
    ]
    config = {"response_format": {"type": "json_object"}}
    output = chat(messages, model = "gpt-4o", config=config, )
    try:
        return json.loads(output)["question"]
    except json.JSONDecodeError:
        print("Error decoding JSON")
    return []

In [109]:
import neo4j
from typing import Literal



class Text2Cypher:
    def __init__(self, driver: neo4j.Driver):
        self.driver = driver
        self.dynamic_sections = {}
        self.required_sections = ["question"]
        self.prompt_template = prompt_template

        schema_string = get_schema(driver)
        self.set_prompt_section("schema", schema_string)

    def set_prompt_section(
        self,
        section: Literal["terminology", "examples", "schema", "question"],
        value: str,
    ):
        self.dynamic_sections[section] = value

    def get_full_prompt(self):
        prompt = self.prompt_template["static"]["instructions"]
        # loop through the prompt_template["dynamic"] and add the values from self.dynamic_sections
        for section in self.prompt_template["dynamic"]:
            if section in self.dynamic_sections:
                prompt += self.prompt_template["dynamic"][section].format(
                    self.dynamic_sections[section]
                )
        return prompt

    def generate_cypher(self):
        # check if required sections are set
        for section in self.required_sections:
            if section not in self.dynamic_sections:
                raise ValueError(
                    f"Section {section} is required to generate a prompt. Use set_prompt_section to set it."
                )
        prompt = self.get_full_prompt()
        cypher = chat(messages=[{"role": "user", "content": prompt}])
        return cypher


prompt_template = {
    "static": {
        "instructions": """
    Instrucciones:
    Genera una sentencia Cypher para consultar una base de datos de grafos y obtener los datos necesarios para responder la pregunta del usuario indicada a continuación.
    Eres un sistema de auditoría automática de documentación y facturación de siniestros vehiculares enviados por un taller a una aseguradora.
    Tu objetivo es analizar, verificar y detectar inconsistencias entre los documentos del siniestro, los daños reportados y los valores facturados.
    Documentos que puedes recibir:
    Debes considerar y cruzar información entre los siguientes documentos:
    Orden de ingreso del vehículo
    Parte o reporte del accidente
    Informe técnico del taller
    Facturas de reparación
    Fotografías del daño
    Presupuesto aprobado

    Instrucciones de formato:
    No incluyas explicaciones ni disculpas en tus respuestas.
    No respondas a ninguna pregunta que solicite algo distinto de construir una sentencia Cypher.
    No incluyas ningún texto adicional aparte de la sentencia Cypher generada.
    RESPONDE ÚNICAMENTE CON CYPHER, SIN BLOQUES DE CÓDIGO.
    Asegúrate de nombrar las variables en RETURN según lo solicitado en la pregunta del usuario.
    """
    },
    "dynamic": {
        "schema": """
    Esquema de la Base de Datos de Grafos:
    Utiliza únicamente los tipos de relaciones y propiedades proporcionados en el esquema.
    No utilices ningún otro tipo de relación o propiedad que no esté incluido en el esquema proporcionado.

    {}
    """,
        "terminology": """
    Mapeo de terminología:
    Esta sección es útil para establecer un mapeo de la terminología entre la pregunta del usuario y el esquema de la base de datos de grafos.
    Definicio de los nodos:
    Marca: Representa la marca comercial del vehículo registrada en la aseguradora.
    Modelo: Representa el modelo específico del vehículo registrado en la aseguradora.
    Siniestro: Representa el tipo de accidente o evento registrado que la asegurador asocia con presuntas reparaciones.
    Item: Representa repuestos, reparaciones, o servicios asociados al siniestro.
    Factura: Representa el documento económico generado por la reparación y que tiene la información del total de la reparación y los datos de la mécanica.
    DetalleFactura: Representa cada línea o ítem dentro de la factura.
    OrdenIngreso: Representa el documento principal de ingreso del vehículo al taller y nodo principla para realcionadar el resto de los nodos.
    Vehiculo: Representa el vehículo involucrado en el accidente y reparación.
    Accidente: Representa el accidente asociado al vehículo repotado por el propietario.
    Propietario: Es el dueño del vehiculo que sufrio el accidente y que reporta el accidente.
    Dano: Representa los daños registrados en el vehículo como consecuencia del accidente.
    Imagen:Representa imágenes asociadas al accidente y utilizadas para análisis automático.
    Deteccion:Representa daños detectados automáticamente mediante IA al procesar las imagenes incluye un grado de confianza de la evaluación.
    {}
    """,
        "examples": """
    Ejemplos:
    Los siguientes ejemplos proporcionan patrones útiles para consultar la base de datos de grafos.
    {}
    """,
        "question": """
    User question: {}
    """,
    },
}

In [110]:

answer_given_description = {
    "type": "function",
    "function": {
        "name": "respond",
        "description": "Si la conversación ya contiene una respuesta completa a la pregunta, utiliza esta herramienta para extraerla. Además, si el usuario inicia una conversación informal, utiliza esta herramienta para recordarle que eres un Auditor Agéntico de Facturación de Siniestros Vehiculares – Verificación Automática de Documentación y Detección de Inconsistencias.",
        "parameters": {
            "type": "object",
            "properties": {
                "answer": {
                    "type": "string",
                    "description": "Responda directamente con la respuesta.",
                }
            },
            "required": ["answer"],
        },
    },
}


def answer_given(answer: str):
    """Extract the answer from a given text."""
    return answer


text2cypher_description = {
    "type": "function",
    "function": {
        "name": "text2cypher",
        "description": "Consulta la base de datos con una pregunta del usuario. Si otras herramientas no son adecuadas, recurre a esta.",
        "parameters": {
            "type": "object",
            "properties": {
                "question": {
                    "type": "string",
                    "description": "La pregunta del usuario para encontrar la respuesta a",
                }
            },
            "required": ["question"],
        },
    },
}


def text2cypher(question: str):
    """Query the database with a user question."""
    t2c = Text2Cypher(neo4j_driver)
    t2c.set_prompt_section("question", question)
    cypher = t2c.generate_cypher()
    try:
        records, _, _ = neo4j_driver.execute_query(cypher)
        return [record.data() for record in records]
    except Exception as e:
        return [f"{cypher} cause an error: {e}"]

siniestralidad_reportada_description = {
    "type": "function",
    "function": {
        "name": "siniestralidad_reportada",
        "description": "Esta consulta construye una vista consolidada de siniestro vehicular, diferentes fuentes de información que incluye el evento del accidente (siniestralidad), vehículo involucrado, accidente asociado , daños reportados (Dano), partes del vehiculo detectadas con imágenes del accidente y  registro de facturación.",
        "parameters": {
            "type": "object",
            "properties": {
                "orderId": {
                    "type": "string",
                    "description": "Identificador de la orden de ingreso",
                }
            },
            "required": ["orderId"],
        },
    },
}


def siniestralidad_reportada(orderId: str):
    """siniestralidad_reportada by orderId."""
    query = """
    MATCH (v:Vehiculo)-[:INGRESA]->(o:OrdenIngreso{ordenId:$orderId})
    MATCH (v)-[:INVOLUCRADO_EN]->(a:Accidente)
    OPTIONAL MATCH (a)-[:REPORTA]->(d:Dano)
    OPTIONAL MATCH (a)-[:TIENE_IMAGEN]->(i:Imagen)-[:TIENE_DETECCION]->(de:Deteccion)
    MATCH (f:Factura)-[:BASADA_EN]->(o)
    MATCH (f)-[:CONTIENE]->(det:DetalleFactura)
    RETURN DISTINCT
        v.modelo AS modelo,
        v.marca AS marca,
        a.descripcion AS descripcion_accidente,
        collect(DISTINCT d.nombre) AS reporte_accidente,
        collect(DISTINCT de.descripcion) AS reporte_imagen,
        collect(DISTINCT det.descripcion) AS reporte_factura
    """
    records, _, _ = neo4j_driver.execute_query(query, orderId=orderId)
    print("siniestralidad_reportada",records)
    return [record.data() for record in records]


diferencias_precios_description = {
    "type": "function",
    "function": {
        "name": "diferencias_precios",
        "description": "Lo que se facturó corresponde realmente a lo autorizado por la aseguradora. Sirve para detectar diferencias de precios, cobros superiores, inconsistencias financieras. Utiliza como parametro la orden de ingreso.",
        "parameters": {
            "type": "object",
            "properties": {
                "orderId": {
                    "type": "string",
                    "description": "Identificador de la orden de ingreso",
                }
            },
            "required": ["orderId"],
        },
    },
}


def diferencias_precios(orderId: str):
    """diferencias_precios by orderId."""
    query = """
    MATCH (f:Factura)-[:CONTIENE]->(df:DetalleFactura), (f)-[:BASADA_EN]->(oi:OrdenIngreso{ordenId:$orderId}), (f)-[:PARA_MODELO]->(m:Modelo),
    (m)-[:SUFRE]->(s:Siniestro) WITH f, oi, m, s, collect({descripcion: df.descripcion, cobrado: df.precio}) AS items_factura
    MATCH (s)-[rel:INCLUYE]->(i:Item) WITH f, oi, m, s, i, rel, items_factura
    UNWIND items_factura AS item WITH f, oi, item, i, rel, 1-apoc.text.jaroWinklerDistance(apoc.text.clean(toLower(item.descripcion)),
    apoc.text.clean(toLower(i.nombre))) AS score WHERE score > 0.65
    RETURN f.facId AS factura, oi.ordenId AS orden, item.descripcion AS item_factura, i.nombre AS item_aseguradora,
    item.cobrado AS cobrado, rel.precio AS acordado ORDER BY item_factura,item_aseguradora DESC
    """
    records, _, _ = neo4j_driver.execute_query(query, orderId=orderId)
    print("diferencias_precios",records)
    return [record.data() for record in records]


detalles_factura_description = {
    "type": "function",
    "function": {
        "name": "diferencias_precios",
        "description": "Muestra la lista de detalles que corresponde a la factura de un vehículo.",
        "parameters": {
            "type": "object",
            "properties": {
                "orderId": {
                    "type": "string",
                    "description": "Identificador de la orden de ingreso",
                }
            },
            "required": ["orderId"],
        },
    },
}


def detalles_factura(orderId: str):
    """diferencias_precios by orderId."""
    query = """
    MATCH (f:Factura)-[:CONTIENE]->(df:DetalleFactura), (f)-[:BASADA_EN]->(oi:OrdenIngreso{ordenId:$orderId})
    RETURN f.facId AS factura, oi.ordenId AS orden, df.descripcion AS item_factura
    ORDER BY item_factura DESC
    """
    records, _, _ = neo4j_driver.execute_query(query, orderId=orderId)
    print("detalles_factura",records)
    return [record.data() for record in records]

In [111]:
tool_picker_prompt = """
    Tu tarea consiste en elegir la herramienta adecuada para responder a la pregunta del usuario.
    Las herramientas disponibles se muestran en el enunciado.
    Asegúrate de proporcionar los argumentos correctos y completos a la herramienta seleccionada.
"""

tools = {
    "siniestralidad_reportada": {
        "description": siniestralidad_reportada_description,
        "function": siniestralidad_reportada
    },
    "diferencias_precios": {
        "description": diferencias_precios_description,
        "function": diferencias_precios
    },
    "detalles_factura": {
        "description": detalles_factura_description,
        "function": detalles_factura
    },
    "text2cypher": {
        "description": text2cypher_description,
        "function": text2cypher
    },
    "answer_given": {
        "description": answer_given_description,
        "function": answer_given
    }
}

def handle_tool_calls(tools: dict[str, any], llm_tool_calls: list[dict[str, any]]):
    output = []
    if llm_tool_calls:
        for tool_call in llm_tool_calls:
            function_to_call = tools[tool_call.function.name]["function"]
            function_args = json.loads(tool_call.function.arguments)
            res = function_to_call(**function_args)
            output.append(res)
    return output



In [112]:
def route_question(question: str, tools: dict[str, any], answers: list[dict[str, str]]):
    llm_tool_calls = tool_choice(
        [
            {
                "role": "system",
                "content": tool_picker_prompt,
            },
            *answers,
            {
                "role": "user",
                "content": f"La pregunta del usuario para encontrar una herramienta que la responda: '{question}'",
            },
        ],
        model = "gpt-4o",
        tools=[tool["description"] for tool in tools.values()],
    )
    return handle_tool_calls(tools, llm_tool_calls)

def handle_user_input(input: str, answers: list[dict[str, str]] = []):
    updated_question = query_update(input, answers)
    response  = route_question(updated_question, tools, answers)
    answers.append({"role": "assistant", "content": f"For the question: '{updated_question}', we have the answer: '{json.dumps(response)}'"})
    return answers

In [113]:
answer_critique_prompt = """

    Eres un experto en identificar si las preguntas han sido respondidas completamente o si existe la oportunidad de enriquecer la respuesta.

    El usuario formulará una pregunta y tú revisarás la información proporcionada para determinar si la pregunta está respondida.

    Si falta algo en la respuesta, proporcionarás un conjunto de nuevas preguntas que se pueden formular para obtener la información faltante.

    Todas las nuevas preguntas deben ser completas, atómicas y específicas.

    Sin embargo, si la información proporcionada es suficiente para responder la pregunta original, responderás con una lista vacía.

    Plantilla JSON para buscar información faltante:

    {
    "questions": ["pregunta1", "pregunta2"]

    }
    """

def critique_answers(question: str, answers: list[dict[str, str]]) -> list[str]:
    messages = [
        {
            "role": "system",
            "content": answer_critique_prompt,
        },
        *answers,
        {
            "role": "user",
            "content": f"La pregunta original del usuario para responder: {question}",
        },
    ]
    config = {"response_format": {"type": "json_object"}}
    output = chat(messages, model="gpt-4o", config=config)
    print(output)
    try:
        return json.loads(output)["questions"]
    except json.JSONDecodeError:
        print("Error decoding JSON")
    return []

In [114]:
main_prompt = """

    Tu trabajo consiste en ayudar al usuario con sus preguntas.
    Recibirás las preguntas del usuario y la información necesaria para responderlas.
    Si falta información para responder a una parte o a la totalidad de la pregunta, deberás indicarlo.
    Solo utilizarás la información proporcionada en el enunciado para responder a las preguntas.
    No está permitido inventar información ni utilizar información externa.
    Tu trabajo consiste en actuar como un asistente encargado de auditar información y responder preguntas del usuario relacionadas con procesos de revisión y validación de datos.

    Recibirás preguntas del usuario junto con la información necesaria para responderlas. Debes analizar cuidadosamente dicha información antes de generar una respuesta.

    Tu objetivo principal es ayudar en la verificación y auditoría de documentos, facturas y datos asociados a procesos de un taller y una aseguradora. En particular, el sistema debe ser capaz de:

    Verificar que los insumos y honorarios cobrados correspondan al tarifario acordado.
    Validar que los costos sean coherentes con la siniestralidad reportada.
    Detectar posibles discrepancias, inconsistencias o cobros duplicados.
    Identificar errores o información faltante en los datos proporcionados.

    Si falta información para responder total o parcialmente a una pregunta, debes indicarlo explícitamente.

    Reglas importantes:
    Utiliza únicamente la información proporcionada en el enunciado o en los datos suministrados.
    No inventes información ni utilices fuentes externas.
    Si no es posible llegar a una conclusión, debes indicarlo claramente.
"""

def main(input: str):
    answers = handle_user_input(input)
    critique = critique_answers(input, answers)
    print(f"Critica: {critique}")
    if critique:
        answers = handle_user_input(" ".join(critique), answers)

    llm_response = chat(
        [
            {"role": "system", "content": main_prompt},
            *answers,
            {"role": "user", "content": f"La pregunta del usuario a responder: {input}"},
        ],
        model="gpt-4o",
    )

    return llm_response

In [115]:
response = main("Valida que los costos sean coherentes con la siniestralidad reportada para la orden de ingreso OI-2026-1088 ?")
print(f"Respuesta: {response}")

siniestralidad_reportada [<Record modelo='RAV4' marca='Toyota' descripcion_accidente='El vehículo frenó intempestivamente debido al tráfico de la zona y fue impactado en la parte posterior por otro automotor que no alcanzó a detenerse, desplazándolo hacia adelante y haciéndolo rozar lateralmente contra la baranda de seguridad.' reporte_accidente=['GUARDACHOQUE POSTERIOR', 'PUERTAS', 'GUARDAFANGO', 'LUCES POSTERIORES'] reporte_imagen=['abolladura-capo', 'abolladura-parachoques-del', 'abolladura-puerta-exterior'] reporte_factura=['Revisión Sistema Eléctrico', 'Espejo Retrovisor', 'Kit de Grapas y Clips', 'Parachoques Delantero', 'Ajuste Mecánico de Bisagras', 'Puerta Delantera']>]
diferencias_precios [<Record factura='FAC088' orden='OI-2026-1088' item_factura='Ajuste Mecánico de Bisagras' item_aseguradora='Ajuste Mecánico de Bisagras' cobrado=280.8 acordado=156.0>, <Record factura='FAC088' orden='OI-2026-1088' item_factura='Espejo Retrovisor' item_aseguradora='Espejo Retrovisor' cobrado=

In [116]:
next_res = main("¿Existen discrepancias significativas entre los costos cobrados y acordados para cada elemento para la orden de ingreso OI-2026-1088? Indica los montos y procentaje de diferencia. ")
print(f"Next response: {next_res}")

{
    "questions": [
        "¿Cuál es el costo total cobrado para cada elemento de la orden de ingreso OI-2026-1088 según la factura FAC088?",
        "¿Cuál es el costo total acordado para cada elemento de la orden de ingreso OI-2026-1088 según la factura FAC088?",
        "¿Qué elementos específicos presentan diferencias significativas entre los costos cobrados y acordados?",
        "¿Cuál es el porcentaje de diferencia entre los costos cobrados y acordados para cada elemento de la orden de ingreso OI-2026-1088?",
        "¿Qué criterios se utilizan para determinar si una discrepancia es significativa?"
    ]
}
Critica: ['¿Cuál es el costo total cobrado para cada elemento de la orden de ingreso OI-2026-1088 según la factura FAC088?', '¿Cuál es el costo total acordado para cada elemento de la orden de ingreso OI-2026-1088 según la factura FAC088?', '¿Qué elementos específicos presentan diferencias significativas entre los costos cobrados y acordados?', '¿Cuál es el porcentaje de dif

In [118]:
next_res = main("Verifica si existen valores duplicados en la factura correspondiente a la orden de ingreso OI-2026-1088.")
print(f"Next response: {next_res}")

{
    "questions": [
        "¿Cuáles son los ítems específicos de la factura FAC088 para la orden de ingreso OI-2026-1088?",
        "¿Existen ítems repetidos en la factura FAC088 para la orden de ingreso OI-2026-1088?",
        "¿Cuántas veces se repite cada ítem en la factura FAC088 para la orden de ingreso OI-2026-1088?",
        "¿Qué criterios se utilizan para determinar si un ítem está duplicado en la factura FAC088 para la orden de ingreso OI-2026-1088?"
    ]
}
Critica: ['¿Cuáles son los ítems específicos de la factura FAC088 para la orden de ingreso OI-2026-1088?', '¿Existen ítems repetidos en la factura FAC088 para la orden de ingreso OI-2026-1088?', '¿Cuántas veces se repite cada ítem en la factura FAC088 para la orden de ingreso OI-2026-1088?', '¿Qué criterios se utilizan para determinar si un ítem está duplicado en la factura FAC088 para la orden de ingreso OI-2026-1088?']
Next response: Para verificar si existen valores duplicados en la factura correspondiente a la orden